In [ ]:
# ==============================================================
# SIMPLE MULTI-TASK, MULTI-STEP AI AGENT USING GEMINI
# Google Colab - Single Cell
# ==============================================================

!pip install -q -U google-genai

import os
from google import genai
from google.genai import types
from getpass import getpass

In [ ]:
# ============================================================
# 3. GEMINI SETUP
# ============================================================

# In Google Colab:
# Left panel -> Secrets
# Add:
#
# GEMINI_API_KEY

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [ ]:
# ==============================================================
# STEP 2: CREATE TOOLS
# ==============================================================

def multiply(a: float, b: float) -> float:
    """
    Multiply two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        Multiplication result
    """

    result = a * b

    print("\n🔧 TOOL USED: multiply")
    print(f"   {a} × {b} = {result}")

    return result


def calculate_percentage(amount: float, percentage: float) -> float:
    """
    Calculate percentage of an amount.

    Args:
        amount: Base amount
        percentage: Percentage to calculate

    Returns:
        Calculated percentage amount
    """

    result = amount * percentage / 100

    print("\n🔧 TOOL USED: calculate_percentage")
    print(f"   {percentage}% of {amount} = {result}")

    return result


def add(a: float, b: float) -> float:
    """
    Add two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        Addition result
    """

    result = a + b

    print("\n🔧 TOOL USED: add")
    print(f"   {a} + {b} = {result}")

    return result


def create_summary(
    item: str,
    quantity: int,
    base_cost: float,
    tax: float,
    total_cost: float
) -> str:
    """
    Create a short purchase summary.

    Args:
        item: Item being purchased
        quantity: Number of items
        base_cost: Cost before tax
        tax: Tax amount
        total_cost: Final total cost

    Returns:
        Purchase summary
    """

    print("\n🔧 TOOL USED: create_summary")

    summary = f"""
PURCHASE SUMMARY

Item       : {item}
Quantity   : {quantity}
Base Cost  : ₹{base_cost:,.2f}
GST        : ₹{tax:,.2f}
Total Cost : ₹{total_cost:,.2f}
"""

    return summary


In [ ]:
# ==============================================================
# STEP 3: DEFINE THE AGENT
# ==============================================================

def run_agent(user_request):

    print("\n" + "=" * 65)
    print("🤖 GEMINI MULTI-TASK / MULTI-STEP AGENT")
    print("=" * 65)

    print("\nUSER REQUEST:")
    print(user_request)

    print("\nAgent is analyzing the task...\n")


    # ----------------------------------------------------------
    # Gemini can automatically decide which Python functions
    # should be called.
    # ----------------------------------------------------------

    response = client.models.generate_content(

        model="gemini-3.5-flash",

        contents=user_request,

        config=types.GenerateContentConfig(

            system_instruction="""
You are a multi-task, multi-step AI agent.

Your job is to solve the user's request step-by-step.

You have access to several tools:

1. multiply
2. calculate_percentage
3. add
4. create_summary

Break complicated requests into multiple steps.

Use tools whenever calculations or actions are required.

For purchase calculations:

Step 1:
Calculate base cost.

Step 2:
Calculate tax/GST.

Step 3:
Calculate final cost.

Step 4:
Create a summary.

Do not skip necessary steps.

After completing all tool calls, provide a clear explanation
of what you did and the final result.
""",

            tools=[
                multiply,
                calculate_percentage,
                add,
                create_summary
            ],

            # Allows Gemini Python SDK to make several tool calls
            # during the task.
            automatic_function_calling=
                types.AutomaticFunctionCallingConfig(
                    maximum_remote_calls=10
                )
        )
    )


    print("\n" + "=" * 65)
    print("✅ FINAL AGENT ANSWER")
    print("=" * 65)

    print(response.text)


# ==============================================================
# STEP 4: RUN THE AGENT
# ==============================================================

question = """
I want to purchase 25 laptops.

Each laptop costs ₹55,000.

Calculate:

1. Total laptop cost
2. GST at 18%
3. Final amount including GST
4. Give me a short purchase summary
"""

run_agent(question)


🤖 GEMINI MULTI-TASK / MULTI-STEP AGENT

USER REQUEST:

I want to purchase 25 laptops.

Each laptop costs ₹55,000.

Calculate:

1. Total laptop cost
2. GST at 18%
3. Final amount including GST
4. Give me a short purchase summary


Agent is analyzing the task...


🔧 TOOL USED: multiply
   25 × 55000 = 1375000

🔧 TOOL USED: calculate_percentage
   18% of 1375000 = 247500.0

🔧 TOOL USED: add
   1375000 + 247500 = 1622500

🔧 TOOL USED: create_summary

✅ FINAL AGENT ANSWER
Here is the breakdown of your laptop purchase calculation:

1. **Total Laptop Cost (Base Cost):** ₹1,375,000  
   *Calculated by multiplying 25 laptops by ₹55,000 each.*

2. **GST at 18%:** ₹247,500  
   *Calculated as 18% of the base cost (₹1,375,000).*

3. **Final Amount Including GST:** ₹1,622,500  
   *Calculated by adding the GST (₹247,500) to the base cost (₹1,375,000).*

4. **Purchase Summary:**
```text
PURCHASE SUMMARY

Item       : laptop
Quantity   : 25
Base Cost  : ₹1,375,000.00
GST        : ₹247,500.00
Total C